In [2]:
import sys, os
print(sys.executable)
print(os.getcwd())

c:\Users\THARUSHA\Documents\diabetes-project\diabetes-project\venv\Scripts\python.exe
c:\Users\THARUSHA\Documents\diabetes-project\diabetes-project


In [5]:
import pandas as pd

df = pd.read_csv("data/diabetic_data.csv", na_values=["?"], low_memory=False)
print(df.shape)
df.isna().sum()


(101766, 50)


encounter_id                    0
patient_nbr                     0
race                         2273
gender                          0
age                             0
weight                      98569
admission_type_id               0
discharge_disposition_id        0
admission_source_id             0
time_in_hospital                0
payer_code                  40256
medical_specialty           49949
num_lab_procedures              0
num_procedures                  0
num_medications                 0
number_outpatient               0
number_emergency                0
number_inpatient                0
diag_1                         21
diag_2                        358
diag_3                       1423
number_diagnoses                0
max_glu_serum               96420
A1Cresult                   84748
metformin                       0
repaglinide                     0
nateglinide                     0
chlorpropamide                  0
glimepiride                     0
acetohexamide 

In [6]:
clean = df.copy()
clean = clean.drop(columns=["weight", "payer_code", "examide", "citoglipton"])
print(clean.shape)


(101766, 46)


In [7]:
clean["readmitted_30"] = (clean["readmitted"] == "<30").astype(int)
print(clean["readmitted_30"].mean())

0.11159915885462728


In [8]:
age_map = {"[0-10)": 5, "[10-20)": 15, "[20-30)": 25, "[30-40)": 35, "[40-50)": 45,
           "[50-60)": 55, "[60-70)": 65, "[70-80)": 75, "[80-90)": 85, "[90-100)": 95}
clean["age_mid"] = clean["age"].map(age_map)
print(clean["age_mid"].isna().sum())

0


In [9]:
def diag_group(code):
    if pd.isna(code):
        return "Unknown"
    code = str(code)
    if code[0] in ("V", "E"):
        return "Other"
    v = float(code)
    if 390 <= v <= 459 or v == 785: return "Circulatory"
    if 460 <= v <= 519 or v == 786: return "Respiratory"
    if 520 <= v <= 579 or v == 787: return "Digestive"
    if int(v) == 250: return "Diabetes"
    if 800 <= v <= 999: return "Injury"
    if 710 <= v <= 739: return "Musculoskeletal"
    if 580 <= v <= 629 or v == 788: return "Genitourinary"
    if 140 <= v <= 239: return "Neoplasms"
    return "Other"

for col in ["diag_1", "diag_2", "diag_3"]:
    clean[col + "_group"] = clean[col].apply(diag_group)

clean["diag_1_group"].value_counts()

diag_1_group
Circulatory        30437
Other              18172
Respiratory        14423
Digestive           9475
Diabetes            8757
Injury              6974
Genitourinary       5117
Musculoskeletal     4957
Neoplasms           3433
Unknown               21
Name: count, dtype: int64

In [10]:
clean.to_csv("cleaned/diabetes_clean.csv", index=False)
print(clean.shape)

(101766, 51)


In [11]:
clean = clean[clean["gender"] != "Unknown/Invalid"]
clean = clean[~clean["discharge_disposition_id"].isin([11, 13, 14, 19, 20, 21])]
print(clean.shape)

(99340, 51)


In [12]:
clean[["race", "medical_specialty", "A1Cresult", "max_glu_serum"]].isna().sum()

race                  2232
medical_specialty    48614
A1Cresult            82506
max_glu_serum        94188
dtype: int64

In [15]:
clean["race"] = clean["race"].fillna("Unknown")
clean["medical_specialty"] = clean["medical_specialty"].fillna("Unknown")
clean["A1Cresult"] = clean["A1Cresult"].fillna("Not tested")
clean["max_glu_serum"] = clean["max_glu_serum"].fillna("Not tested")

In [16]:
clean.to_csv("cleaned/diabetes_clean.csv", index=False)
print(clean.shape)

(99340, 51)


In [17]:
print(clean.shape)
print(clean[["race", "medical_specialty", "A1Cresult", "max_glu_serum"]].isna().sum().sum())

(99340, 51)
0


In [18]:
import sqlite3

conn = sqlite3.connect("db/hospital.db")
conn.executescript("""
DROP VIEW IF EXISTS v_encounters_full;
DROP TABLE IF EXISTS encounters;
DROP TABLE IF EXISTS patients;

CREATE TABLE patients (
    patient_nbr INTEGER PRIMARY KEY,
    race TEXT,
    gender TEXT
);

CREATE TABLE encounters (
    encounter_id INTEGER PRIMARY KEY,
    patient_nbr INTEGER NOT NULL,
    age_group TEXT, age_mid INTEGER,
    admission_type_id INTEGER, discharge_disposition_id INTEGER, admission_source_id INTEGER,
    time_in_hospital INTEGER, medical_specialty TEXT,
    num_lab_procedures INTEGER, num_procedures INTEGER, num_medications INTEGER,
    number_outpatient INTEGER, number_emergency INTEGER, number_inpatient INTEGER,
    number_diagnoses INTEGER, max_glu_serum TEXT, A1Cresult TEXT,
    change TEXT, diabetesMed TEXT, diag_1_group TEXT,
    readmitted TEXT, readmitted_30 INTEGER,
    FOREIGN KEY (patient_nbr) REFERENCES patients(patient_nbr)
);
""")
print("Tables created")

Tables created


In [19]:
def first_known(series):
    known = series[series != "Unknown"]
    return known.iloc[0] if len(known) else "Unknown"

patients = (
    clean.sort_values("encounter_id")
    .groupby("patient_nbr")
    .agg(race=("race", first_known), gender=("gender", "first"))
    .reset_index()
)
patients.to_sql("patients", conn, if_exists="append", index=False)
print(len(patients))

69987


In [20]:
enc = clean.rename(columns={"age": "age_group"})[[
    "encounter_id", "patient_nbr", "age_group", "age_mid",
    "admission_type_id", "discharge_disposition_id", "admission_source_id",
    "time_in_hospital", "medical_specialty",
    "num_lab_procedures", "num_procedures", "num_medications",
    "number_outpatient", "number_emergency", "number_inpatient",
    "number_diagnoses", "max_glu_serum", "A1Cresult",
    "change", "diabetesMed", "diag_1_group",
    "readmitted", "readmitted_30"]]
enc.to_sql("encounters", conn, if_exists="append", index=False)
print(len(enc))

99340


In [21]:
import pandas as pd

lookups, current = {}, None
with open("data/IDS_mapping.csv", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line.startswith(("admission_type_id", "discharge_disposition_id", "admission_source_id")):
            current = line.split(",")[0]
            lookups[current] = []
        elif line and line != "," and current:
            key, _, desc = line.partition(",")
            lookups[current].append((int(key), desc.strip().strip('"')))

table_names = {"admission_type_id": "admission_type",
               "discharge_disposition_id": "discharge_disposition",
               "admission_source_id": "admission_source"}

for id_col, rows in lookups.items():
    pd.DataFrame(rows, columns=[id_col, "description"]).to_sql(
        table_names[id_col], conn, if_exists="replace", index=False)
    print(table_names[id_col], len(rows))

admission_type 8
discharge_disposition 30
admission_source 25


In [22]:
conn.executescript("""
DROP VIEW IF EXISTS v_encounters_full;
CREATE VIEW v_encounters_full AS
SELECT e.*, p.race, p.gender,
       t.description AS admission_type,
       s.description AS admission_source,
       d.description AS discharge_disposition
FROM encounters e
JOIN patients p ON e.patient_nbr = p.patient_nbr
LEFT JOIN admission_type t ON e.admission_type_id = t.admission_type_id
LEFT JOIN admission_source s ON e.admission_source_id = s.admission_source_id
LEFT JOIN discharge_disposition d ON e.discharge_disposition_id = d.discharge_disposition_id;
""")
conn.commit()

pd.read_sql("SELECT COUNT(*) AS rows_in_view FROM v_encounters_full", conn)

,rows_in_view
0,99340


In [23]:
query = """
SELECT diag_1_group,
       COUNT(*) AS visits,
       ROUND(AVG(readmitted_30) * 100, 1) AS readmit_pct
FROM encounters
GROUP BY diag_1_group
ORDER BY readmit_pct DESC
"""
pd.read_sql(query, conn)

,diag_1_group,visits,readmit_pct
0,Unknown,20,25.0
1,Diabetes,8661,13.1
2,Injury,6851,12.4
3,Other,17793,11.7
4,Circulatory,29680,11.7
5,Genitourinary,5002,11.0
6,Neoplasms,3131,10.9
7,Digestive,9333,10.8
8,Respiratory,13934,10.1
9,Musculoskeletal,4935,9.5


In [24]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("db/hospital.db")
data = pd.read_sql("SELECT * FROM v_encounters_full", conn)
print(data.shape)

(99340, 28)


In [25]:
num_cols = ["age_mid", "time_in_hospital", "num_lab_procedures", "num_procedures",
            "num_medications", "number_outpatient", "number_emergency",
            "number_inpatient", "number_diagnoses"]
cat_cols = ["gender", "race", "admission_type", "discharge_disposition",
            "medical_specialty", "diag_1_group", "A1Cresult", "max_glu_serum",
            "change", "diabetesMed"]

X = pd.get_dummies(data[num_cols + cat_cols], columns=cat_cols)
y = data["readmitted_30"]
print(X.shape)

(99340, 141)


In [27]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=data["patient_nbr"]))

print(len(train_idx), len(test_idx))

79567 19773


In [28]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200, max_depth=12, min_samples_leaf=20,
    class_weight="balanced", random_state=42, n_jobs=-1)

model.fit(X.iloc[train_idx], y.iloc[train_idx])
print("Training finished")

Training finished


In [29]:
from sklearn.metrics import roc_auc_score

probs = model.predict_proba(X.iloc[test_idx])[:, 1]
print("ROC-AUC:", round(roc_auc_score(y.iloc[test_idx], probs), 3))

ROC-AUC: 0.674


In [30]:
importance = pd.Series(model.feature_importances_, index=X.columns)
importance.sort_values(ascending=False).head(10)

number_inpatient                                                                                           0.280820
discharge_disposition_Discharged to home                                                                   0.091979
number_emergency                                                                                           0.058732
discharge_disposition_Discharged/transferred to another rehab fac including rehab units of a hospital .    0.055146
number_diagnoses                                                                                           0.050427
num_medications                                                                                            0.049294
time_in_hospital                                                                                           0.046842
num_lab_procedures                                                                                         0.040134
age_mid                                                                 

In [31]:
pred = pd.DataFrame({
    "encounter_id": data.loc[test_idx, "encounter_id"].values,
    "actual_readmitted_30": y.iloc[test_idx].values,
    "predicted_probability": probs,
})
pred["risk_band"] = pd.qcut(pred["predicted_probability"], 3,
                            labels=["Low", "Medium", "High"]).astype(str)

pred.groupby("risk_band")["actual_readmitted_30"].mean().round(3)

risk_band
High      0.189
Low       0.054
Medium    0.097
Name: actual_readmitted_30, dtype: float64

In [33]:
pred.to_sql("predictions", conn, if_exists="replace", index=False)
conn.commit()

pred.to_csv("cleaned/predictions.csv", index=False)
data.to_csv("cleaned/encounters_full.csv", index=False)

print(pred.shape)

(19773, 4)
